In [1]:
import polars as pl

# **LLM Fine-Tuning**

### __Objective:__

In this demo, you will fine-tune the Tiny LLama 1B using Parameter-Efficient Fine-Tuning (PEFT) with LoRA.
You will tokenize a dataset and configure key LoRA parameters (rank, scaling, dropout) for efficient training.
Finally, compare the model outputs before and after fine-tuning to showcase the method's effectiveness.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Building LLM Applications**.
- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.
- Refer to Lesson_01 **Demo_01_Zero_Shot_Prompting.ipynb** Step 1 for creating a virtual environment and installing the requirements.txt
- Ensure you select the right kernel **Python (myenv)** while running the demos
---


### **Steps to be followed:**

1. Install required packages and import libraries
2. Set device and load pre-trained model
3. Configure PEFT with LoRA
4. Move the model to the device
5. Load and preprocess the dataset
6. Define a custom data collator
7. Generate and store model outputs before fine-tuning
8. Configure training arguments and fine-tune the model
9. Compare model outputs after fine-tuning

---

### **Step 1: Install required packages and import libraries**

- Install and import essential libraries for model loading, dataset management, and fine-tuning

Installing and importing packages like transformers, peft, and datasets prepares the environment for loading pre-trained models, applying parameter-efficient fine-tuning, and managing datasets. This sets the foundation for the entire demo.

In [2]:
# !pip uninstall -y torch torchvision torchaudio transformers accelerate peft datasets bitsandbytes numpy
!pip install torch --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.40.2
!pip install peft==0.10.0
!pip install datasets bitsandbytes sentencepiece einops scipy
# !pip install --upgrade bitsandbytes>=0.46.1
!pip install bitsandbytes --upgrade
!pip install accelerate --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages t

In [3]:
!pip install accelerate --upgrade

In [4]:
!pip install -q "transformers>=4.41.0,<6.0.0" accelerate bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 58.9 MB/s eta 0:00:00


In [5]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [6]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    default_data_collator
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

### **Step 2: Set device and load pre-trained model**

- Determine if a GPU is available and load the pre-trained Tiny Llama 1B model along with its tokenizer

Using a GPU (if available) significantly accelerates training. Loading a pre-trained model like Tiny LLama 1B gives us a base model that we can fine-tune, saving both time and resources compared to training from scratch.

In [7]:
# Set device to GPU if available (e.g., T4), otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
from transformers import BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # ✅ use the lighter version for stability

# Quantization config — uses much less VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # 4-bit loading (big memory saver)
    bnb_4bit_compute_dtype=torch.float16,  # compute safely in fp16
    bnb_4bit_quant_type="nf4", #normalfloat4 for internal torch formatting
)

# Load model with automatic layer distribution
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0},   # manually put full model on GPU 0
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model.config.use_cache = False  # Important for training
model.eval()

print(f"✅ Model loaded successfully on: {model.device}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✅ Model loaded successfully on: cuda:0


In [9]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm(

### **Step 3: Configure PEFT with LoRA**

- We now set up the Low-Rank Adaptation (LoRA) configuration for PEFT. This involves defining several parameters:

    - `r=8`: This is the LoRA rank. It determines the size of the low-rank matrices added to the model. A higher rank can capture more nuances but increases parameters slightly.
    - `lora_alpha=32`: This scaling factor adjusts the magnitude of the LoRA weights. It helps in stabilizing training by scaling the low-rank updates.
    - `target_modules=["q_proj", "v_proj"]`: Specifies which layers to apply LoRA to. For Tiny Llama 1B, targeting the *q_proj and v_proj* module (the attention projection layer) is common since these layers significantly impact the model's performance.
    - `lora_dropout=0.1`: This dropout rate is used on the LoRA layers to regularize the training and prevent overfitting.
    - `bias="none"`: Indicates that the bias parameters are not being fine-tuned.
    - `task_type="CAUSAL_LM"`: Specifies that our task is causal language modeling.


This step is crucial because it configures the PEFT method. By adding trainable low-rank matrices only to key components, we significantly reduce the number of parameters to update, making fine-tuning both memory- and compute-efficient.

In [10]:


lora_config = LoraConfig(
    r=4,                      # smaller rank for low memory
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # ✅ Tiny Llama-specific layers
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)


The Tiny Llama 1B model is now wrapped with LoRA-based PEFT. Only the additional low-rank parameters in the targeted modules will be updated during fine-tuning, making the process more efficient while retaining performance.

### **Step 4: Move the model to the device**

- Move the model to the GPU (if available) to ensure faster training and inference


Transferring the model to the correct device (GPU/CPU) is essential for performance. GPUs, in particular, accelerate the matrix operations involved in training deep neural networks.

In [11]:
# Move the model to the chosen device (GPU)
# model = model.to(device)

### **Step 5: Load and preprocess the dataset**

- Loading the 5% of a Stanford Alpaca Dataset
- Tokenize the text using the Tiny Llama 1B tokenizer with a maximum sequence length of 128 tokens
- Filter out any examples that yield empty token sequences


Preprocessing the data is a critical step before training. Tokenizing converts raw text into numerical tokens that the model can understand. Filtering ensures that only valid data is passed to the model, which improves training quality.

In [12]:
# Load a subset of Stanford Alpaca for demonstration
dataset = load_dataset("tatsu-lab/alpaca", split="train[:5%]")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) > 0)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2600 [00:00<?, ? examples/s]

In [13]:
dataset['text'][0]

'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'

### **Step 6: Define a custom data collator**

- Define a custom data collator function to prepare batches of data for training. This function ensures:
    - The `input_ids` are cast to long tensors.
    - The `labels` are set correctly. If labels aren’t provided, they are set to be the same as the input_ids.


A data collator is used to combine individual examples into a batch. This ensures that all sequences in the batch have consistent formatting, which is critical for training stability and performance.

In [14]:
def collate_fn(features):
    batch = default_data_collator(features)
    batch["input_ids"] = batch["input_ids"].long()
    batch["labels"] = batch["input_ids"].clone()
    print(batch)
    return batch


### **Step 7: Generate and store model outputs before fine-tuning**
- Generate outputs for several predefined prompts
- These outputs represent the baseline performance of the model in its pre-fine-tuned state and are stored for later comparison.

Capturing the model’s behavior before fine-tuning is crucial for demonstrating how PEFT changes the model's responses. This baseline is essential for a clear before-and-after comparison.

In [16]:

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

prompts = [
    "Artificial intelligence is",
    "Quantum computing works by",
    "Renewable energy affects the environment because"
]

model.config.use_cache = False

print("\n=== Generating outputs BEFORE fine-tuning ===")
before_outputs = {}
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            use_cache=False,        # ✅ prevents Tiny Llama cache bug
            max_new_tokens=64,      # ✅ shorter output for low VRAM
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            num_return_sequences=1,
            repetition_penalty=1.7
        )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    before_outputs[prompt] = text
    print(f"\nPrompt: {prompt}\nBefore: {text}\n{'-'*60}")



=== Generating outputs BEFORE fine-tuning ===

Prompt: Artificial intelligence is
Before: Artificial intelligence is the ability of machines to perform tasks similar in nature, or as close as possible. The field has gained significant popularity over time with advances such…
A Brief Overview Of AI And Its Implications For Education: Introduction To Advanced Robotics In Engineering By Chad Rice Free eBooks
------------------------------------------------------------

Prompt: Quantum computing works by
Before: Quantum computing works by simulating the behavior of quantum particles, such as electrons or atoms. These simulations involve measuring and manipulating entangled pairs (qubits) with other quits in a way that enables complex computations to be performed rapidly for certain applications like encryption, cryptography etc., which can only work on classical computers
------------------------------------------------------------

Prompt: Renewable energy affects the environment because

### **Step 8: Configure training arguments and fine-tune the model**

We now set up the training configuration with specific parameters:

- `output_dir`: Directory to save the fine-tuned model
- `run_name`: Name for the training run
- `max_steps`: Limits the number of training steps (set to 50 for this demo)
- `per_device_train_batch_size`: Batch size for each device
- `learning_rate`: The learning rate for fine-tuning
- `logging_steps and save_steps`: Frequency of logging and saving checkpoints
- `num_train_epochs`: Number of training epochs
- `fp16`: Enables mixed precision training for speed on GPUs
- `no_cuda`: Indicates that CUDA (GPU) should be used if available


Then we initialize the Hugging Face Trainer with our model, training arguments, dataset, and data collator, and run the training process.

This step fine-tunes the model using our dataset and LoRA configuration. The training arguments control the fine-tuning process, and running the trainer updates only the low-rank parameters introduced by LoRA.

In [17]:

print("\n⏳ Fine-tuning Tiny-LLama-1B with LoRA...")

training_args = TrainingArguments(
    output_dir="./tiny_llama_1b_lora_finetuned",
    max_steps=50,                        # ✅ small training steps
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=20,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=collate_fn,
)

trainer.train()
print("✅ Fine-tuning complete!")



⏳ Fine-tuning Tiny-LLama-1B with LoRA...
{'input_ids': tensor([[    1, 13866,   338,   385, 15278,   393, 16612,   263,  3414, 29892,
          3300,  2859,   411,   385,  1881,   393,  8128,  4340,  3030, 29889,
         14350,   263,  2933,   393,  7128,  2486,  1614,  2167,   278,  2009,
         29889,    13,    13,  2277, 29937,  2799,  4080, 29901,    13,  7648,
          1598,   278,  1134,   310,  6035,  1304,   297,   278,  1494,  2958,
           446, 29889,    13,    13,  2277, 29937, 10567, 29901,    13, 29984,
         29901,  1724,  1258,   278,  2646,   412,  1827,   746,   372,   471,
         28996,   373, 29973,    13, 29909, 29901,  9531,   448,   372,   925,
          1235,   714,   263,  2217,   377,   457, 29889,    13,    13,  2277,
         29937, 13291, 29901,    13,  1576,  2958,   446,   338,   385,  1342,
           310,   263,  6035,  2729,   373,  3632,  3021,  2873, 29889,   450,
          2958,   446,   338,   773,   278,  3632,  3021,  2873,   376, 298

### **Step 9: Compare model outputs after fine-tuning**

- Generate outputs for the same set of prompts using the fine-tuned model
- These outputs are compared side-by-side with the baseline outputs generated earlier.


Comparing outputs before and after fine-tuning clearly demonstrates the impact of the training process. It shows how the model's responses change after being fine-tuned with PEFT, providing practical insights into the effectiveness of this method.

In [18]:

print("\n=== Comparing Outputs BEFORE and AFTER Fine-Tuning ===")
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output_ids = model.generate(
        **inputs,
        use_cache=False,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        num_return_sequences=2,
        repetition_penalty=1.7
    )
    after_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(f"\nPrompt: {prompt}")
    print("| Before Fine-Tuning |")
    print(before_outputs[prompt])
    print("\n| After Fine-Tuning |")
    print(after_text)
    print("=" * 80)



=== Comparing Outputs BEFORE and AFTER Fine-Tuning ===

Prompt: Artificial intelligence is
| Before Fine-Tuning |
Artificial intelligence is the ability of machines to perform tasks similar in nature, or as close as possible. The field has gained significant popularity over time with advances such…
A Brief Overview Of AI And Its Implications For Education: Introduction To Advanced Robotics In Engineering By Chad Rice Free eBooks

| After Fine-Tuning |
Artificial intelligence is changing the way we live and work. It's used to create personalized products, manage supply chains more efficiently than human labor alone can do or even plan complex events such as games in a video game console with deep learning algorithms . In recent years it has become increasingly popular among consumers due its benefits like

Prompt: Quantum computing works by
| Before Fine-Tuning |
Quantum computing works by simulating the behavior of quantum particles, such as electrons or atoms. These simulations invol

### __Conclusion__

By following these steps, you have successfully fine-tuned the Tine Llama model using Parameter-Efficient Fine-Tuning (PEFT) with LoRA. You learned how to configure LoRA parameters such as rank, scaling factor, and dropout to train only the most impactful low-rank layers instead of the entire model. Using a subset of Stanford Alpaca, you efficiently adapted the model with minimal computational overhead.

The comparison of outputs before and after fine-tuning clearly illustrates how PEFT enhances the model’s contextual reasoning and response quality. This approach demonstrates that LoRA provides a cost-effective and scalable strategy for fine-tuning large models while maintaining their strong baseline capabilities.

---